In [1]:

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
# from robustness_experiment_box.analysis.report_creator import ReportCreator
import re
import scipy.stats as st


In [13]:
df = pd.read_csv('results_CFX_ijcai_dec.csv', index_col=0)
print(df.columns)

metrics = ['IM1', 'IM2', 'correctness', 'l1', 'l2', 'linf', 'implausibility']

Index(['network', 'image', 'original_label', 'original_predicted_label',
       'predicted_label', 'target', 'method', 'IM1', 'IM2', 'correctness',
       'l1', 'l2', 'linf', 'implausibility', 'optim_time', 'faithfulness',
       'weak correctness'],
      dtype='object')


In [16]:
aggregated = df.groupby('method').agg({
    'IM1': 'mean',
    'IM2': 'mean',
    'correctness': 'sum',  # Sum 'correctness'
    'l1': 'mean',
    'l2': 'mean',
    'linf': 'mean',
    'implausibility': 'mean'
})
print(aggregated.to_latex(index=True))

\begin{tabular}{lrrrrrrr}
\toprule
 & IM1 & IM2 & correctness & l1 & l2 & linf & implausibility \\
method &  &  &  &  &  &  &  \\
\midrule
C-Min-Edit & 3.085697 & 0.128633 & 3670 & 178.073945 & 15.743924 & 1.991621 & 388.593219 \\
Captum-MinParamPerturbation & 0.995004 & 0.114611 & 0 & 189.054333 & 11.210082 & 0.999974 & 470.072163 \\
Min-Edit & 3.212955 & 0.128588 & 3298 & 179.106782 & 15.816440 & 1.992767 & 390.353605 \\
PIECE & 2.298321 & 0.116996 & 1455 & 167.546775 & 14.986852 & 1.982975 & 394.597832 \\
alibi-CF & 1.599614 & 0.110882 & 3237 & 30.502378 & 5.407858 & 1.805593 & 417.874625 \\
alibi-Proto-CF & 1.075957 & 0.253185 & 3593 & 585.158522 & 23.714568 & 1.783340 & 766.760977 \\
\bottomrule
\end{tabular}



In [ ]:

from scipy.stats import spearmanr


def scatter_metrics(df):
    plt.figure(figsize=(12,12))
    sns.set(font_scale = 2)
    sns.set_style({'font.family':'serif', 'font.serif':'Times New Roman'})

    # Create a pairplot
    
    # Create the pairplot
    g = sns.pairplot(df, vars=metrics, diag_kind='kde', corner=True,hue='method', palette='deep')

    for i, j in zip(*np.tril_indices_from(g.axes, -1)):
        ax = g.axes[i, j]
        x = df[metrics[j]]
        y = df[metrics[i]]
        # Calculate Spearman correlation
        r, _ = spearmanr(x, y)
        # Annotate the correlation on the plot
        ax.annotate(f"ρ={r:.2f}", xy=(0.5, 0.9), xycoords='axes fraction', ha='center', fontsize=10, color='red')

    # Save the figure
    plt.savefig(f"test.png")
    plt.close()


scatter_metrics(df)

In [ ]:

def boxplot_per_method(df):

    # Create a boxplot for each metric
    for metric in metrics:
        plt.figure(figsize=(12,12))
        sns.set(font_scale = 2)
        sns.set_style({'font.family':'serif', 'font.serif':'Times New Roman'})
        sns.set_palette('deep')

        
        # Create the boxplot
        sns.boxplot(
            data=df,
            x='method',
            y=metric,
            hue = 'method',
            legend = False
        )
        
        # Add titles and labels
        plt.xlabel('Method')
        plt.ylabel(metric)
        plt.xticks(rotation=45, ha='right')
    
        # Save the plot
        plt.savefig(f"figures/boxplot/boxplot_{metric}.pdf", dpi=300, bbox_inches='tight')
        plt.close()
boxplot_per_method(df)

In [ ]:
import itertools
def scatter_plot_per_metric(df):
    plt.figure(figsize=(12,12))
    sns.set(font_scale = 2)
    sns.set_style({'font.family':'serif', 'font.serif':'Times New Roman'})
    # Select the relevant columns for the scatterplots
    hues = ['original_label', 'target', 'network', 'method']
    # Create scatterplots for each combination of metrics
    for x_metric, y_metric in itertools.combinations(metrics, 2):
        for hue in hues:

    
            scatter = sns.scatterplot(
                data=df, 
                x=x_metric, 
                y=y_metric, 
                hue=hue,
                palette='deep', 
                alpha=0.8
            )
            
            # Calculate and annotate Spearman correlation
            r, _ = spearmanr(df[x_metric], df[y_metric])
            plt.annotate(
                f"ρ={r:.2f}", 
                xy=(0.05, 0.95), 
                xycoords='axes fraction', 
                fontsize=12, 
                color='red', 
                ha='left', 
                va='top'
            )
            
            # Add plot labels and title
            plt.title(f'{y_metric} vs {x_metric}', fontsize=14)
            plt.xlabel(x_metric)
            plt.ylabel(y_metric)
            plt.legend(fontsize='small', title_fontsize='small')
            # Save the figure to a file
            filename = f"figures/scatter/scatter_{y_metric}_vs_{x_metric}_in_{hue}.png"
            plt.savefig(filename, dpi=300, bbox_inches='tight')
            plt.close()


scatter_plot_per_metric(df)
